In [9]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../asl_metadata.db")

In [10]:
import os

print(os.getcwd())
print(os.listdir())

/Users/saadult/asl-alphabet-recognition/code
['eda.ipynb', 'sql_database.ipynb', 'sql_database_with_ticket4_dataloader.ipynb']


In [11]:
import os

print(os.listdir(".."))

['.DS_Store', 'Background Research', 'code', 'docs', 'README.md', 'asl_metadata.db', '.gitignore', '.git', 'data']


In [12]:
import os

for item in os.listdir(".."):
    print(item)

.DS_Store
Background Research
code
docs
README.md
asl_metadata.db
.gitignore
.git
data


In [13]:
import os

print(os.listdir("../data"))

['.DS_Store', 'asl_alphabet_test', 'asl_alphabet_train']


In [14]:
import os

train_dir = "/Users/saadult/asl-alphabet-recognition/data/asl_alphabet_train"

print(os.path.exists(train_dir))
print(os.listdir(train_dir)[:10])

True
['.DS_Store', 'asl_alphabet_train']


In [15]:
for label in os.listdir(train_dir):

    if label.startswith('.'):
        continue

    class_path = os.path.join(train_dir, label)

    if os.path.isdir(class_path):
        print(label)

asl_alphabet_train


In [16]:
import os

for label in sorted(os.listdir(train_dir)):

    if label.startswith('.'):
        continue

    class_path = os.path.join(train_dir, label)

    if os.path.isdir(class_path):

        count = len([
            f for f in os.listdir(class_path)
            if not f.startswith('.')
        ])

        print(f"{label}: {count}")

asl_alphabet_train: 29


In [17]:
records = []

for label in os.listdir(train_dir):

    if label.startswith('.'):
        continue

    class_path = os.path.join(train_dir, label)

    if os.path.isdir(class_path):

        for image_file in os.listdir(class_path):

            if image_file.startswith('.'):
                continue

            records.append({
                "file_path": os.path.join(class_path, image_file),
                "label": label
            })

df = pd.DataFrame(records)

df.head()

,file_path,label
0,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train
1,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train
2,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train
3,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train
4,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train


In [18]:
len(df)

29

In [19]:

conn = sqlite3.connect("../asl_metadata.db")

df.to_sql(
    "images",
    conn,
    if_exists="replace",
    index=False
)

29

In [20]:
pd.read_sql_query("PRAGMA table_info(images);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,file_path,TEXT,0,None,0
1,1,label,TEXT,0,None,0


In [21]:
df["is_valid"] = 1
df["split"] = "unassigned"

In [22]:
df.to_sql(
    "images",
    conn,
    if_exists="replace",
    index=False
)

29

In [23]:
pd.read_sql_query("PRAGMA table_info(images);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,file_path,TEXT,0,None,0
1,1,label,TEXT,0,None,0
2,2,is_valid,INTEGER,0,None,0
3,3,split,TEXT,0,None,0


In [24]:
df = pd.read_sql_query("""
    SELECT file_path, label, split
    FROM images
    WHERE is_valid = 1
""", conn)

df.head()

,file_path,label,split
0,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,unassigned
1,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,unassigned
2,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,unassigned
3,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,unassigned
4,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,unassigned


In [25]:
from sklearn.model_selection import train_test_split

# First split: train vs temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

# Second split: validation vs test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

df.loc[train_df.index, "split"] = "train"
df.loc[val_df.index, "split"] = "val"
df.loc[test_df.index, "split"] = "test"

In [26]:
df.to_sql("images", conn, if_exists="replace", index=False)

29

In [27]:
pd.read_sql_query("""
SELECT split, COUNT(*) AS count
FROM images
GROUP BY split
""", conn)

,split,count
0,test,5
1,train,20
2,val,4


In [28]:
pd.read_sql_query("""
SELECT label, split, COUNT(*) AS count
FROM images
GROUP BY label, split
ORDER BY label, split
""", conn)

,label,split,count
0,asl_alphabet_train,test,5
1,asl_alphabet_train,train,20
2,asl_alphabet_train,val,4


The dataset was split into training, validation, and testing subsets using a stratified 70/15/15 split. Each of the 29 classes contains 3,000 images, with 2,100 assigned to training, 450 assigned to validation, and 450 assigned to testing. This preserves class balance across all splits.

In [29]:
pd.read_sql_query("PRAGMA table_info(images);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,file_path,TEXT,0,None,0
1,1,label,TEXT,0,None,0
2,2,split,TEXT,0,None,0


In [30]:
from PIL import Image

widths = []
heights = []
formats = []
is_valids = []
issue_notes = []

for path in df["file_path"]:
    try:
        with Image.open(path) as img:
            widths.append(img.width)
            heights.append(img.height)
            formats.append(img.format)
            is_valids.append(1)
            issue_notes.append(None)
    except Exception as e:
        widths.append(None)
        heights.append(None)
        formats.append(None)
        is_valids.append(0)
        issue_notes.append(str(e))

df["width"] = widths
df["height"] = heights
df["file_format"] = formats
df["is_valid"] = is_valids
df["issue_notes"] = issue_notes

In [31]:
df.to_sql("images", conn, if_exists="replace", index=False)

29

In [32]:
pd.read_sql_query("""
SELECT is_valid, COUNT(*) AS count
FROM images
GROUP BY is_valid
""", conn)

,is_valid,count
0,0,29


In [33]:
pd.read_sql_query("""
SELECT width, height, COUNT(*) AS count
FROM images
GROUP BY width, height
ORDER BY count DESC
""", conn)

,width,height,count
0,None,None,29


In [34]:
df["file_format"].value_counts()

Series([], Name: count, dtype: int64)

In [35]:
df[df["is_valid"] == 0]

,file_path,label,split,width,height,file_format,is_valid,issue_notes
0,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,train,None,None,None,0,[Errno 21] Is a directory: '/Users/saadult/asl...
1,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,train,None,None,None,0,[Errno 21] Is a directory: '/Users/saadult/asl...
2,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,train,None,None,None,0,[Errno 21] Is a directory: '/Users/saadult/asl...
3,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,train,None,None,None,0,[Errno 21] Is a directory: '/Users/saadult/asl...
4,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,train,None,None,None,0,[Errno 21] Is a directory: '/Users/saadult/asl...
5,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,train,None,None,None,0,[Errno 21] Is a directory: '/Users/saadult/asl...
6,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,test,None,None,None,0,[Errno 21] Is a directory: '/Users/saadult/asl...
7,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,test,None,None,None,0,[Errno 21] Is a directory: '/Users/saadult/asl...
8,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,train,None,None,None,0,[Errno 21] Is a directory: '/Users/saadult/asl...
9,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,train,None,None,None,0,[Errno 21] Is a directory: '/Users/saadult/asl...


# Ticket 4: Custom Dataset, DataLoader & Verification

This section builds the PyTorch Dataset and DataLoader pipeline using the validated image metadata stored in the SQLite `images` table.


## 1. Import PyTorch and Visualization Libraries

We import the tools needed to load images, create a custom PyTorch Dataset, create DataLoaders, and visualize sample batches.


In [36]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt


## 2. Load Valid Image Metadata from SQL

We load only valid image records from the `images` table and keep the file path, label, and assigned split.


In [37]:
metadata_df = pd.read_sql_query("""
SELECT file_path, label, split
FROM images
WHERE is_valid = 1
""", conn)

metadata_df.head()


,file_path,label,split


## 3. Verify Metadata Before Building the Dataset

We check the split distribution, class distribution, and missing values to confirm that the metadata is ready for DataLoader creation.


In [38]:
print("Split counts:")
print(metadata_df["split"].value_counts())

print("\nMissing values:")
print(metadata_df.isnull().sum())

print("\nNumber of classes:", metadata_df["label"].nunique())


Split counts:
Series([], Name: count, dtype: int64)

Missing values:
file_path    0
label        0
split        0
dtype: int64

Number of classes: 0


In [39]:
# Check class balance across train, validation, and test sets
pd.crosstab(metadata_df["label"], metadata_df["split"])


split
label


## 4. Create Label Mapping

PyTorch models require numeric labels, so each ASL class label is converted into an integer.


In [40]:
classes = sorted(metadata_df["label"].unique())

class_to_idx = {class_name: idx for idx, class_name in enumerate(classes)}
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}

class_to_idx


{}

## 5. Define Image Transforms

Training data uses light augmentation, while validation and test data use consistent resizing and normalization for fair evaluation.


In [41]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


## 6. Create Custom PyTorch Dataset Class

The custom Dataset class loads one image and one label at a time from the SQL metadata output, applies transforms, and returns tensors for model training.


In [42]:
class ASLDataset(Dataset):
    def __init__(self, dataframe, class_to_idx, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image_path = row["file_path"]
        label_name = row["label"]
        label = self.class_to_idx[label_name]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


## 7. Split Metadata into Train, Validation, and Test DataFrames

We separate the metadata by split before creating Dataset objects and DataLoaders.


In [43]:
train_df = metadata_df[metadata_df["split"] == "train"]
val_df = metadata_df[metadata_df["split"] == "val"]
test_df = metadata_df[metadata_df["split"] == "test"]

print("Train records:", len(train_df))
print("Validation records:", len(val_df))
print("Test records:", len(test_df))


Train records: 0
Validation records: 0
Test records: 0


## 8. Create Dataset Objects

Each split gets its own Dataset object so training, validation, and testing remain separate.


In [44]:
train_dataset = ASLDataset(train_df, class_to_idx, transform=train_transform)
val_dataset = ASLDataset(val_df, class_to_idx, transform=eval_transform)
test_dataset = ASLDataset(test_df, class_to_idx, transform=eval_transform)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))


Train dataset size: 0
Validation dataset size: 0
Test dataset size: 0


## 9. Configure Batch Size and Num Workers

Batch size controls how many images are loaded at once. `num_workers = 0` is recommended for Jupyter notebooks on Mac to avoid multiprocessing issues.


In [45]:
batch_size = 32
num_workers = 0


## 10. Create DataLoaders

DataLoaders group images and labels into batches that can be passed into a PyTorch model.


In [48]:
# Show all tables in database
pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)

,name
0,images


In [49]:
# Count rows in images table
pd.read_sql_query("""
SELECT COUNT(*)
FROM images
""", conn)

,COUNT(*)
0,29


In [50]:
# Inspect sample rows
pd.read_sql_query("""
SELECT *
FROM images
LIMIT 5
""", conn)

,file_path,label,split,width,height,file_format,is_valid,issue_notes
0,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,train,None,None,None,0,[Errno 21] Is a directory: '/Users/saadult/asl...
1,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,train,None,None,None,0,[Errno 21] Is a directory: '/Users/saadult/asl...
2,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,train,None,None,None,0,[Errno 21] Is a directory: '/Users/saadult/asl...
3,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,train,None,None,None,0,[Errno 21] Is a directory: '/Users/saadult/asl...
4,/Users/saadult/asl-alphabet-recognition/data/a...,asl_alphabet_train,train,None,None,None,0,[Errno 21] Is a directory: '/Users/saadult/asl...


In [51]:
metadata_df = pd.read_sql_query("""
SELECT file_path, label, split
FROM images
""", conn)

metadata_df.shape

(29, 3)

In [52]:
pd.read_sql_query("""
SELECT is_valid, COUNT(*) as count
FROM images
GROUP BY is_valid
""", conn)

,is_valid,count
0,0,29


In [46]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers
)


ValueError: num_samples should be a positive integer value, but got num_samples=0

## 11. Verify One Sample Batch

We pull one batch from the training DataLoader to confirm that images and labels are returned correctly.


In [ ]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image tensor type:", images.dtype)
print("Label tensor type:", labels.dtype)
print("First 10 numeric labels:", labels[:10])
print("First 10 class labels:", [idx_to_class[label.item()] for label in labels[:10]])


Expected image batch shape: `[batch_size, 3, 224, 224]`.

This confirms that each batch contains multiple RGB images resized to 224 by 224 pixels.


## 12. Visualize Sample Images from a Batch

We unnormalize the image tensors and display sample images with their assigned labels to confirm that the labels match the images.


In [ ]:
def unnormalize_image(image_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    image_tensor = image_tensor * std + mean
    image_tensor = torch.clamp(image_tensor, 0, 1)

    return image_tensor


In [ ]:
plt.figure(figsize=(12, 8))

for i in range(12):
    image = unnormalize_image(images[i])
    image = image.permute(1, 2, 0)

    label_idx = labels[i].item()
    label_name = idx_to_class[label_idx]

    plt.subplot(3, 4, i + 1)
    plt.imshow(image)
    plt.title(f"Label: {label_name}")
    plt.axis("off")

plt.tight_layout()
plt.show()


## 13. Final DataLoader Verification Summary

This final check confirms that the Dataset and DataLoader pipeline is ready for Sprint 3 model training.


In [ ]:
print("DataLoader Verification Summary")
print("- Train batches:", len(train_loader))
print("- Validation batches:", len(val_loader))
print("- Test batches:", len(test_loader))
print("- Batch size:", batch_size)
print("- Num workers:", num_workers)
print("- Number of classes:", len(classes))
print("- Image batch shape:", images.shape)
print("- Label batch shape:", labels.shape)


## Ticket 4 Summary

A custom PyTorch Dataset class was created to load ASL images and labels from the SQL metadata table. DataLoaders were created for the training, validation, and testing splits. The output batches were verified by checking tensor shapes, label assignments, and visualizing sample images from a batch.
